In [ ]:
import pandas as pd
import numpy as np
import os

# Paths
OUT_DIR = r"C:\Users\HP\Downloads\swarmbench_task\swarmbench_task\environment\input_artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

NUM_FILES = 10
ROWS_PER_FILE = 500000 
UNIQUE_COUNT = 100000 # The "True" number of unique customers

print(f"Generating {NUM_FILES * ROWS_PER_FILE * 2:,} total rows across 20 files...")

# 1. Create a Master "Source of Truth" for Unique Customers
master_customers = pd.DataFrame({
    'CustomerId': range(1000000, 1000000 + UNIQUE_COUNT),
    'Surname': [f"User_{i}" for i in range(UNIQUE_COUNT)],
    'Geography': np.random.choice(['France', 'Spain', 'Germany', 'FRA', 'French', 'DE'], UNIQUE_COUNT),
    'Gender': np.random.choice(['Male', 'Female', 'F', 'M'], UNIQUE_COUNT),
    'Age': np.random.randint(18, 90, UNIQUE_COUNT),
    'Tenure': np.random.randint(0, 11, UNIQUE_COUNT)
})

master_accounts = pd.DataFrame({
    'CustomerId': range(1000000, 1000000 + UNIQUE_COUNT),
    'CreditScore': np.random.randint(300, 850, UNIQUE_COUNT),
    'Balance': np.random.uniform(0, 250000, UNIQUE_COUNT).round(2),
    'NumOfProducts': np.random.randint(1, 5, UNIQUE_COUNT),
    'HasCrCard': np.random.choice(['Yes', 'No', 1, 0], UNIQUE_COUNT),
    'IsActiveMember': np.random.choice(['Yes', 'No', 1, 0], UNIQUE_COUNT),
    'EstimatedSalary': np.random.uniform(10000, 200000, UNIQUE_COUNT).round(2),
    'Exited': np.random.choice([0, 1], UNIQUE_COUNT)
})

# 2. Generate Shards with Duplicates and "Dirty" Formatting
for i in range(1, NUM_FILES + 1):
    # Sample with heavy replacement to create duplicates across files
    cust_shard = master_customers.sample(ROWS_PER_FILE, replace=True).copy()
    acc_shard = master_accounts.sample(ROWS_PER_FILE, replace=True).copy()
    
    # Introduce "Dirty" Currency Strings (matches your PDF analysis [cite: 18, 197])
    cust_shard['EstimatedSalary_Dirty'] = cust_shard.merge(
        master_accounts[['CustomerId', 'EstimatedSalary']], on='CustomerId'
    )['EstimatedSalary'].apply(lambda x: f"€{x:,}")
    
    acc_shard['Balance'] = acc_shard['Balance'].apply(lambda x: f"€{x}")

    # Save Shards
    cust_shard.to_csv(os.path.join(OUT_DIR, f"customer_shard_{i}.csv"), index=False)
    acc_shard.to_csv(os.path.join(OUT_DIR, f"account_shard_{i}.csv"), index=False)
    
    print(f"Done: Shard {i}/{NUM_FILES} (1 Million rows generated in this step)")

print("\n🚀 Adversarial Scale-Level Shards Generated successfully.")